# Study 01: Dataset Exploration & Canonical Statistics
**Goal:** Understand the Liver Tumor dataset — intensity distributions, tumor coverage, class imbalance, and save canonical statistics for reporting.

## 1. Setup

In [ ]:
import sys, json, random, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
import torch
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
import pandas as pd
pd.set_option('display.max_columns', 10)
warnings.filterwarnings('ignore')

## 2. Load Volume Index

In [ ]:
from src.data_loader import DataPathManager
path_manager = DataPathManager()
volume_index = path_manager.build_index()
print(f'Volumes: {len(volume_index["volumes"])}')
print(f'Total slices: {sum(len(v) for v in volume_index["image_paths"].values())}')

## 3. Dataset Statistics (GPU-Accelerated)

In [ ]:
from src.config import DEVICE
from PIL import Image

n_sample = min(2000, sum(len(v) for v in volume_index['image_paths'].values()))
all_pairs = [(vid, idx) for vid, paths in volume_index['image_paths'].items() for idx in range(len(paths))]
sampled = random.sample(all_pairs, n_sample)

intensities, tumor_pixels, tumor_flags = [], [], []
for vid, sid in tqdm(sampled, desc='Scanning slices'):
    imp = volume_index['image_paths'][vid][sid]
    msk = volume_index['mask_paths'][vid][sid]
    img = np.array(Image.open(imp).convert('L'), dtype=np.float32)
    mask = np.array(Image.open(msk), dtype=np.uint8)
    intensities.append(img)
    tumor_pixels.append((mask > 0).sum())
    tumor_flags.append((mask > 0).any())

intensities = np.concatenate([i.ravel() for i in intensities])
print(f'Intensity: mean={intensities.mean():.2f}, std={intensities.std():.2f}')
print(f'Tumor slices: {sum(tumor_flags)}/{len(tumor_flags)} ({100*sum(tumor_flags)/len(tumor_flags):.2f}%)')
print(f'Avg tumor pixels per positive slice: {np.mean([t for t in tumor_pixels if t>0]):.0f}')

## 4. Intensity Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(intensities[intensities>0].ravel(), bins=100, color='steelblue', alpha=0.7)
axes[0].set_title('Pixel Intensity Distribution (nonzero)')
axes[0].set_xlabel('Intensity'); axes[0].set_ylabel('Count')
axes[1].hist(intensities[intensities>0].ravel(), bins=100, color='steelblue', alpha=0.7, range=(0, 200))
axes[1].axvline(-100, color='red', ls='--', label='HU window low')
axes[1].axvline(400, color='red', ls='--', label='HU window high')
axes[1].set_title('Zoomed (0-200) with HU window')
axes[1].set_xlabel('Intensity'); axes[1].legend()
plt.tight_layout()
plt.savefig(Path.cwd().parent/'figures'/'intensity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tumor Size Distribution

In [ ]:
tumor_sizes = [t for t in tumor_pixels if t > 0]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(tumor_sizes, bins=50, color='crimson', alpha=0.7)
axes[0].set_title('Tumor Size Distribution (pixels per positive slice)')
axes[0].set_xlabel('Tumor pixels'); axes[0].set_ylabel('Count')
axes[1].hist(np.log10([max(s,1) for s in tumor_sizes]), bins=50, color='crimson', alpha=0.7)
axes[1].set_title('Tumor Size (log10 pixels)')
axes[1].set_xlabel('log10(tumor pixels)')
plt.tight_layout(); plt.show()

## 6. Class Imbalance Analysis

In [ ]:
bg_pixels, fg_pixels = [], []
for vid, sid in tqdm(sampled, desc='Computing imbalance'):
    msk = np.array(Image.open(volume_index['mask_paths'][vid][sid]), dtype=np.uint8)
    bg_pixels.append((msk == 0).sum())
    fg_pixels.append((msk > 0).sum())
total_bg, total_fg = sum(bg_pixels), sum(fg_pixels)
ratio = total_bg / max(total_fg, 1)
print(f'Background pixels: {total_bg:,}')
print(f'Foreground (tumor) pixels: {total_fg:,}')
print(f'Imbalance ratio: {ratio:.1f}:1')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(['Background', 'Tumor'], [total_bg, total_fg], color=['lightblue', 'crimson'])
ax.set_ylabel('Total pixels')
ax.set_title(f'Class Imbalance ({ratio:.0f}:1)')
ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
plt.tight_layout(); plt.show()

## 7. Per-Volume Statistics

In [ ]:
vol_stats = {}
for vid in tqdm(volume_index['volumes'], desc='Per-volume'):
    slices = volume_index['image_paths'].get(vid, [])
    n_tumor = 0
    for sidx in range(len(slices)):
        msk = np.array(Image.open(volume_index['mask_paths'][vid][sidx]), dtype=np.uint8)
        if (msk > 0).any(): n_tumor += 1
    vol_stats[vid] = {'n_slices': len(slices), 'n_tumor_slices': n_tumor}

df = pd.DataFrame(vol_stats).T
print(df.describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['n_slices'], bins=30, color='teal', alpha=0.7)
axes[0].set_xlabel('Slices per volume'); axes[0].set_ylabel('Count'); axes[0].set_title('Volume Size')
axes[1].hist(df['n_tumor_slices'], bins=30, color='crimson', alpha=0.7)
axes[1].set_xlabel('Tumor slices per volume'); axes[1].set_ylabel('Count'); axes[1].set_title('Tumor Presence')
plt.tight_layout(); plt.show()

## 8. Save Canonical Statistics

In [ ]:
stats = {
    'n_total_slices': sum(len(v) for v in volume_index['image_paths'].values()),
    'n_volumes': len(volume_index['volumes']),
    'mean_intensity': float(intensities.mean()),
    'std_intensity': float(intensities.std()),
    'tumor_slice_pct': 100 * sum(tumor_flags) / len(tumor_flags),
    'imbalance_ratio': ratio,
    'total_bg_pixels': int(total_bg),
    'total_fg_pixels': int(total_fg),
}
out = Path.cwd().parent / 'data' / 'metadata' / 'statistics.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(stats, indent=2))
print(f'Saved to {out}')
print(json.dumps(stats, indent=2))

## 9. Key Insights

In [ ]:
print('='*60)
print('KEY INSIGHTS')
print('='*60)
print(f'1. Extreme class imbalance: {ratio:.0f}:1 background:tumor')
print(f'2. Only {100*sum(tumor_flags)/len(tumor_flags):.1f}% of slices contain tumor')
print(f'3. FocalLoss recommended (gamma=2.0) to handle imbalance')
print(f'4. HU windowing [-100, 400] captures liver intensity range')
print(f'5. CLAHE preprocessing helps enhance low-contrast tumors')